In [2]:
import os, time, random, numpy as np
import scipy.io as sio
from sklearn.model_selection import train_test_split
from sklearn.decomposition import PCA
from sklearn.metrics import accuracy_score, f1_score, recall_score, precision_score

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
random.seed(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

# -------------------- Data Loading --------------------
def load_dataset(data_path, dataset_name, config):
    file_path = os.path.join(data_path, config['file'])
    data = sio.loadmat(file_path)[config['data_key']]
    if config['file'] == config['gt_file']:
        labels = sio.loadmat(file_path)[config['gt_key']]
    else:
        gt_path = os.path.join(data_path, config['gt_file'])
        labels = sio.loadmat(gt_path)[config['gt_key']]
    if len(labels.shape) == 3:
        labels = labels.squeeze()
    return data, labels

def applyPCA(X, numComponents):
    original_shape = X.shape
    X_reshaped = X.reshape(-1, X.shape[-1])
    pca = PCA(n_components=min(numComponents, X.shape[-1]), whiten=True)
    X_pca = pca.fit_transform(X_reshaped)
    return X_pca.reshape(original_shape[0], original_shape[1], -1)

def createImageCubes(X, y, windowSize=5):
    margin = windowSize // 2
    paddedX = np.pad(X, ((margin, margin), (margin, margin), (0,0)), mode='reflect')
    patches, labels_ = [], []
    for r in range(margin, paddedX.shape[0]-margin):
        for c in range(margin, paddedX.shape[1]-margin):
            if y[r-margin, c-margin] > 0:
                patch = paddedX[r-margin:r+margin+1, c-margin:c+margin+1]
                patches.append(patch)
                labels_.append(y[r-margin, c-margin]-1)
    return np.array(patches), np.array(labels_)

# -------------------- Adaptive Clustering --------------------
class AdaptiveClustering(nn.Module):
    def __init__(self, weight_shape):
        super().__init__()
        self.out_channels = weight_shape[0]
        self.D = int(np.prod(weight_shape[1:]))
        self.keep_ratio = nn.Parameter(torch.tensor(0.5))
        self.tau = nn.Parameter(torch.tensor(1.0))

    @property
    def K(self):
        k = int(max(2, torch.sigmoid(self.keep_ratio).item() * self.out_channels))
        return min(k, self.out_channels)

    def get_centroids(self):
        if not hasattr(self, '_centroids_buffer') or self._centroids_buffer.shape[0] != self.K:
            self.register_buffer('_centroids_buffer', torch.randn(self.K, self.D) * 0.01)
        return self._centroids_buffer

    def forward(self, W):
        W_flat = W.view(W.shape[0], -1)
        O, D = W_flat.shape
        K_eff = self.K
        centroids = self.get_centroids()[:K_eff]
        tau_val = self.tau.abs() + 0.05
        dist = torch.cdist(W_flat, centroids)
        A = F.softmax(-dist / tau_val, dim=1)
        W_compressed = torch.mm(A, centroids)
        return W_compressed.view_as(W), A

# -------------------- Large Compressible SST (all layers) --------------------
class LargeCompressibleSST(nn.Module):
    def __init__(self, n_classes, S, L, patch_size=5, embed_dim=128, num_heads=8, num_layers=4,
                 compress_layers=None, gate_init=0.0):
        super().__init__()
        self.compress_layers = compress_layers if compress_layers is not None else []
        self.compression_modules = nn.ModuleDict()
        self.gates = nn.ParameterDict()
        self.embed_dim = embed_dim
        self.num_layers = num_layers

        self.input_dim = patch_size * patch_size * L
        self.pos_embed = nn.Parameter(torch.randn(1, 1, embed_dim) * 0.02)
        self.token_embed = nn.Linear(self.input_dim, embed_dim)

        self.layers = nn.ModuleList()
        for i in range(num_layers):
            layer = CompressibleTransformerBlock(embed_dim, num_heads, i, self.compress_layers, gate_init)
            self.layers.append(layer)

        self.classifier = nn.Linear(embed_dim, n_classes)
        self._register_compression_modules()

    def _register_compression_modules(self):
        if 'token_embed' in self.compress_layers:
            self.compression_modules['token_embed'] = AdaptiveClustering(self.token_embed.weight.shape)
            self.gates['token_embed'] = nn.Parameter(torch.tensor(0.0))
        for layer in self.layers:
            layer.register_compression_modules(self.compression_modules, self.gates)

    def get_gate(self, name):
        if self.gates is None or name not in self.gates:
            return 0.0
        return torch.sigmoid(self.gates[name])

    def get_compressed_weight(self, name, original_weight):
        if self.compression_modules is None or name not in self.compression_modules:
            return original_weight
        gate = self.get_gate(name)
        if gate < 0.01:
            return original_weight
        compressed, _ = self.compression_modules[name](original_weight)
        return (1 - gate) * original_weight + gate * compressed

    def forward(self, x):
        B = x.shape[0]
        x = x.view(B, -1)
        w_embed = self.get_compressed_weight('token_embed', self.token_embed.weight)
        x = F.linear(x, w_embed, self.token_embed.bias)
        x = x.unsqueeze(1) + self.pos_embed
        for layer in self.layers:
            x = layer(x, self)
        x = x.squeeze(1)
        return self.classifier(x)

    def sparsity_loss(self, lambda_base=3.0):
        if self.gates is None or len(self.gates) == 0:
            return torch.tensor(0.0)
        loss = 0.0
        for name, gate_param in self.gates.items():
            gate = self.get_gate(name)
            loss += lambda_base * (1 - gate)
            if name in self.compression_modules:
                keep = torch.sigmoid(self.compression_modules[name].keep_ratio)
                loss += 1.5 * keep
        return loss / len(self.gates)

    def get_learned_K(self):
        return {name: mod.K for name, mod in self.compression_modules.items()}

class CompressibleTransformerBlock(nn.Module):
    def __init__(self, embed_dim, num_heads, layer_idx, compress_layers, gate_init):
        super().__init__()
        self.layer_idx = layer_idx
        self.embed_dim = embed_dim
        self.compress_layers = compress_layers
        self.q_proj = nn.Linear(embed_dim, embed_dim)
        self.k_proj = nn.Linear(embed_dim, embed_dim)
        self.v_proj = nn.Linear(embed_dim, embed_dim)
        self.out_proj = nn.Linear(embed_dim, embed_dim)
        self.ffn1 = nn.Linear(embed_dim, embed_dim * 4)
        self.ffn2 = nn.Linear(embed_dim * 4, embed_dim)
        self.norm1 = nn.LayerNorm(embed_dim)
        self.norm2 = nn.LayerNorm(embed_dim)
        self.compression_module_names = [f'q_{layer_idx}', f'k_{layer_idx}', f'v_{layer_idx}', f'attn_out_{layer_idx}',
                                         f'ffn1_{layer_idx}', f'ffn2_{layer_idx}']

    def register_compression_modules(self, compression_modules, gates):
        for name in self.compression_module_names:
            if name in self.compress_layers:
                if name.startswith('q_'):
                    weight = self.q_proj.weight
                elif name.startswith('k_'):
                    weight = self.k_proj.weight
                elif name.startswith('v_'):
                    weight = self.v_proj.weight
                elif name.startswith('attn_out_'):
                    weight = self.out_proj.weight
                elif name.startswith('ffn1_'):
                    weight = self.ffn1.weight
                elif name.startswith('ffn2_'):
                    weight = self.ffn2.weight
                else:
                    continue
                compression_modules[name] = AdaptiveClustering(weight.shape)
                gates[name] = nn.Parameter(torch.tensor(0.0))

    def get_compressed_weight(self, model, name, original_weight):
        if name not in model.compression_modules:
            return original_weight
        gate = model.get_gate(name)
        if gate < 0.01:
            return original_weight
        compressed, _ = model.compression_modules[name](original_weight)
        return (1 - gate) * original_weight + gate * compressed

    def forward(self, x, model):
        # Attention
        q_w = self.get_compressed_weight(model, f'q_{self.layer_idx}', self.q_proj.weight)
        k_w = self.get_compressed_weight(model, f'k_{self.layer_idx}', self.k_proj.weight)
        v_w = self.get_compressed_weight(model, f'v_{self.layer_idx}', self.v_proj.weight)
        out_w = self.get_compressed_weight(model, f'attn_out_{self.layer_idx}', self.out_proj.weight)
        q = F.linear(x, q_w, self.q_proj.bias)
        k = F.linear(x, k_w, self.k_proj.bias)
        v = F.linear(x, v_w, self.v_proj.bias)
        attn_out = F.scaled_dot_product_attention(q, k, v)
        attn_out = F.linear(attn_out, out_w, self.out_proj.bias)
        x = self.norm1(x + attn_out)
        # FFN
        w1 = self.get_compressed_weight(model, f'ffn1_{self.layer_idx}', self.ffn1.weight)
        b1 = self.ffn1.bias
        w2 = self.get_compressed_weight(model, f'ffn2_{self.layer_idx}', self.ffn2.weight)
        b2 = self.ffn2.bias
        ffn_out = F.linear(x, w1, b1)
        ffn_out = F.gelu(ffn_out)
        ffn_out = F.linear(ffn_out, w2, b2)
        x = self.norm2(x + ffn_out)
        return x

# -------------------- Training --------------------
def train_two_stage(model, train_loader, X_test_t, y_test_t,
                    epochs_stage1=150, epochs_stage2=100, final_lambda=3.0, lr=0.0005):
    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=1e-5)
    ce_loss = nn.CrossEntropyLoss()
    print("\n--- Stage 1: Training without compression loss ---")
    best_val_acc = 0
    best_state = None
    patience = 30
    no_improve = 0
    for epoch in range(epochs_stage1):
        model.train()
        total_loss = 0.0
        for xb, yb in train_loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            optimizer.zero_grad()
            out = model(xb)
            loss = ce_loss(out, yb)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
        model.eval()
        with torch.no_grad():
            out_val = model(X_test_t)
            val_acc = accuracy_score(y_test_t.cpu(), out_val.argmax(1).cpu())
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
            no_improve = 0
        else:
            no_improve += 1
            if no_improve >= patience and best_val_acc > 0.95:
                print(f"Early stopping at epoch {epoch+1} (best val_acc={best_val_acc:.4f})")
                model.load_state_dict(best_state)
                break
        if (epoch+1) % 20 == 0:
            print(f"Stage1 Epoch {epoch+1}/{epochs_stage1} | Loss: {total_loss/len(train_loader):.4f} | Val Acc: {val_acc:.4f}")
    print("\n--- Stage 2: Adding compression loss (K evolution) ---")
    for epoch in range(epochs_stage2):
        model.train()
        total_loss = 0.0
        lam = final_lambda * (epoch / max(1, epochs_stage2-1))
        for xb, yb in train_loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            optimizer.zero_grad()
            out = model(xb)
            loss = ce_loss(out, yb)
            loss += lam * model.sparsity_loss()
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
        if (epoch+1) % 10 == 0 or epoch == epochs_stage2-1:
            model.eval()
            with torch.no_grad():
                out_val = model(X_test_t)
                val_acc = accuracy_score(y_test_t.cpu(), out_val.argmax(1).cpu())
            gates = {name: model.get_gate(name).item() for name in model.gates}
            k_vals = model.get_learned_K()
            print(f"\n📊 Epoch {epoch+1}/{epochs_stage2} | λ={lam:.3f} | Val Acc={val_acc:.4f}")
            print(f"   Gates: {gates}")
            print(f"   Learned K: {k_vals}")
    return model

def evaluate_full(model, X, y_true):
    model.eval()
    t1 = time.perf_counter()
    with torch.no_grad():
        out = model(X)
    t2 = time.perf_counter()
    pred = out.argmax(dim=1).cpu().numpy()
    y_true = y_true.cpu().numpy()
    return {
        'accuracy': accuracy_score(y_true, pred),
        'precision': precision_score(y_true, pred, average='macro', zero_division=0),
        'recall': recall_score(y_true, pred, average='macro', zero_division=0),
        'f1': f1_score(y_true, pred, average='macro', zero_division=0),
        'inf_time_ms': (t2 - t1) * 1000
    }

# -------------------- Build DKM Compressed Tiny Model (Fully Corrected) --------------------
def build_dkm_tiny_model(original_model, learned_K, input_dim, n_classes, num_layers=4):
    # New embed_dim from token_embed's K
    new_embed_dim = learned_K.get('token_embed', original_model.embed_dim)
    class DKMCompressedSST(nn.Module):
        def __init__(self):
            super().__init__()
            self.embed_dim = new_embed_dim
            self.token_embed = nn.Linear(input_dim, new_embed_dim)
            self.pos_embed = nn.Parameter(torch.randn(1, 1, new_embed_dim) * 0.02)
            self.layers = nn.ModuleList()
            for i in range(num_layers):
                qk_dim = learned_K.get(f'q_{i}', new_embed_dim)
                attn_out_dim = learned_K.get(f'attn_out_{i}', new_embed_dim)
                ffn1_dim = learned_K.get(f'ffn1_{i}', new_embed_dim * 4)
                self.layers.append(nn.ModuleDict({
                    'q': nn.Linear(new_embed_dim, qk_dim),
                    'k': nn.Linear(new_embed_dim, qk_dim),
                    'v': nn.Linear(new_embed_dim, qk_dim),
                    'out': nn.Linear(qk_dim, new_embed_dim),
                    'ffn1': nn.Linear(new_embed_dim, ffn1_dim),
                    'ffn2': nn.Linear(ffn1_dim, new_embed_dim),
                    'norm1': nn.LayerNorm(new_embed_dim),
                    'norm2': nn.LayerNorm(new_embed_dim)
                }))
            self.classifier = nn.Linear(new_embed_dim, n_classes)

        def forward(self, x):
            B = x.shape[0]
            x = x.view(B, -1)
            x = self.token_embed(x)
            x = x.unsqueeze(1) + self.pos_embed
            for layer in self.layers:
                q = layer['q'](x)
                k = layer['k'](x)
                v = layer['v'](x)
                attn_out = F.scaled_dot_product_attention(q, k, v)
                attn_out = layer['out'](attn_out)
                x = layer['norm1'](x + attn_out)
                ffn_out = layer['ffn1'](x)
                ffn_out = F.gelu(ffn_out)
                ffn_out = layer['ffn2'](ffn_out)
                x = layer['norm2'](x + ffn_out)
            x = x.squeeze(1)
            return self.classifier(x)

    tiny = DKMCompressedSST().to(DEVICE)

    with torch.no_grad():
        # Token embed
        if 'token_embed' in original_model.compression_modules:
            mod = original_model.compression_modules['token_embed']
            centroids = mod.get_centroids()
            orig_weight = original_model.token_embed.weight.data
            dist = torch.cdist(orig_weight, centroids)
            assignments = dist.argmin(dim=1)
            reconstructed = centroids[assignments]
            K = mod.K
            tiny.token_embed.weight.data = reconstructed[:K]
            tiny.token_embed.bias.data = original_model.token_embed.bias.data[:K]
        else:
            tiny.token_embed.weight.data = original_model.token_embed.weight.data
            tiny.token_embed.bias.data = original_model.token_embed.bias.data

        for i, orig_layer in enumerate(original_model.layers):
            tiny_layer = tiny.layers[i]
            # Q, K, V
            q_name = f'q_{i}'
            if q_name in original_model.compression_modules:
                mod = original_model.compression_modules[q_name]
                centroids = mod.get_centroids()
                orig_weight = orig_layer.q_proj.weight.data
                dist = torch.cdist(orig_weight, centroids)
                assignments = dist.argmin(dim=1)
                reconstructed = centroids[assignments]
                K = mod.K
                tiny_layer['q'].weight.data = reconstructed[:K, :tiny.embed_dim]
                tiny_layer['q'].bias.data = orig_layer.q_proj.bias.data[:K] if orig_layer.q_proj.bias is not None else None
                tiny_layer['k'].weight.data = tiny_layer['q'].weight.data.clone()
                tiny_layer['k'].bias.data = tiny_layer['q'].bias.data.clone()
                tiny_layer['v'].weight.data = tiny_layer['q'].weight.data.clone()
                tiny_layer['v'].bias.data = tiny_layer['q'].bias.data.clone()
            else:
                tiny_layer['q'].weight.data = orig_layer.q_proj.weight.data
                tiny_layer['q'].bias.data = orig_layer.q_proj.bias.data
                tiny_layer['k'].weight.data = orig_layer.k_proj.weight.data
                tiny_layer['k'].bias.data = orig_layer.k_proj.bias.data
                tiny_layer['v'].weight.data = orig_layer.v_proj.weight.data
                tiny_layer['v'].bias.data = orig_layer.v_proj.bias.data

            # Attention output
            out_name = f'attn_out_{i}'
            if out_name in original_model.compression_modules:
                mod = original_model.compression_modules[out_name]
                centroids = mod.get_centroids()
                orig_weight = orig_layer.out_proj.weight.data
                dist = torch.cdist(orig_weight, centroids)
                assignments = dist.argmin(dim=1)
                reconstructed = centroids[assignments]
                K_out = mod.K
                qk_dim = learned_K.get(f'q_{i}', tiny.embed_dim)
                tiny_layer['out'].weight.data = reconstructed[:K_out, :qk_dim]
                tiny_layer['out'].bias.data = orig_layer.out_proj.bias.data[:K_out]
            else:
                tiny_layer['out'].weight.data = orig_layer.out_proj.weight.data
                tiny_layer['out'].bias.data = orig_layer.out_proj.bias.data

            # FFN1
            ffn1_name = f'ffn1_{i}'
            if ffn1_name in original_model.compression_modules:
                mod = original_model.compression_modules[ffn1_name]
                centroids = mod.get_centroids()
                orig_weight = orig_layer.ffn1.weight.data
                dist = torch.cdist(orig_weight, centroids)
                assignments = dist.argmin(dim=1)
                reconstructed = centroids[assignments]
                K1 = mod.K
                tiny_layer['ffn1'].weight.data = reconstructed[:K1, :tiny.embed_dim]
                tiny_layer['ffn1'].bias.data = orig_layer.ffn1.bias.data[:K1] if orig_layer.ffn1.bias is not None else None
            else:
                tiny_layer['ffn1'].weight.data = orig_layer.ffn1.weight.data
                tiny_layer['ffn1'].bias.data = orig_layer.ffn1.bias.data

            # FFN2
            ffn2_name = f'ffn2_{i}'
            if ffn2_name in original_model.compression_modules:
                mod = original_model.compression_modules[ffn2_name]
                centroids = mod.get_centroids()
                orig_weight = orig_layer.ffn2.weight.data
                dist = torch.cdist(orig_weight, centroids)
                assignments = dist.argmin(dim=1)
                reconstructed = centroids[assignments]
                K2 = mod.K
                ffn1_k = learned_K.get(f'ffn1_{i}', tiny.embed_dim * 4)
                # Output dimension is tiny.embed_dim, input dimension is ffn1_k
                tiny_layer['ffn2'].weight.data = reconstructed[:tiny.embed_dim, :ffn1_k]
                tiny_layer['ffn2'].bias.data = orig_layer.ffn2.bias.data[:tiny.embed_dim]
            else:
                tiny_layer['ffn2'].weight.data = orig_layer.ffn2.weight.data
                tiny_layer['ffn2'].bias.data = orig_layer.ffn2.bias.data

        # Classifier
        tiny.classifier.weight.data = original_model.classifier.weight.data[:, :tiny.embed_dim]
        tiny.classifier.bias.data = original_model.classifier.bias.data

    return tiny

# -------------------- Main Experiment --------------------
def run_experiment(data_path, dataset_name, config, ep_baseline=100, ep_stage2=50, ep_fine=80):
    print("\n" + "="*80)
    print(f"DATASET: {dataset_name}")
    print(f"PCA Components: {config['pca_components']}, Patch Size: {config['patch_size']}")
    print("="*80)

    X, y = load_dataset(data_path, dataset_name, config)
    X_pca = applyPCA(X, config['pca_components'])
    X_patches, y_labels = createImageCubes(X_pca, y, windowSize=config['patch_size'])
    n_classes = len(np.unique(y_labels))
    X_train, X_test, y_train, y_test = train_test_split(X_patches, y_labels, test_size=0.25, stratify=y_labels, random_state=SEED)
    S, _, L = X_train.shape[1], X_train.shape[2], X_train.shape[3]

    train_loader = torch.utils.data.DataLoader(
        torch.utils.data.TensorDataset(torch.tensor(X_train, dtype=torch.float32), torch.tensor(y_train, dtype=torch.long)),
        batch_size=64, shuffle=True)
    X_test_t = torch.tensor(X_test, dtype=torch.float32).to(DEVICE)
    y_test_t = torch.tensor(y_test, dtype=torch.long).to(DEVICE)

    # ---- Baseline (large) ----
    print("\n--- Training Baseline (No Compression) ---")
    baseline = LargeCompressibleSST(n_classes, S, L, patch_size=config['patch_size'],
                                    embed_dim=128, num_heads=8, num_layers=4, compress_layers=[]).to(DEVICE)
    baseline_start = time.time()
    baseline = train_two_stage(baseline, train_loader, X_test_t, y_test_t,
                               epochs_stage1=ep_baseline, epochs_stage2=0, final_lambda=0.0)
    baseline_train_time = time.time() - baseline_start
    baseline_metrics = evaluate_full(baseline, X_test_t, y_test_t)
    baseline_params = sum(p.numel() for p in baseline.parameters())
    baseline_size_mb = baseline_params * 4 / (1024 * 1024)

    # ---- Self‑adaptive compressed model (all layers) ----
    print("\n--- Training Self-Adaptive Compressed Model (all layers) ---")
    compress_candidates = ['token_embed']
    for i in range(4):
        compress_candidates.extend([f'q_{i}', f'k_{i}', f'v_{i}', f'attn_out_{i}', f'ffn1_{i}', f'ffn2_{i}'])
    student = LargeCompressibleSST(n_classes, S, L, patch_size=config['patch_size'],
                                   embed_dim=128, num_heads=8, num_layers=4,
                                   compress_layers=compress_candidates, gate_init=0.0).to(DEVICE)
    student_start = time.time()
    student = train_two_stage(student, train_loader, X_test_t, y_test_t,
                              epochs_stage1=120, epochs_stage2=ep_stage2, final_lambda=3.0, lr=0.0005)
    student_train_time = time.time() - student_start

    learned_K = student.get_learned_K()
    print("\n--- Learned K values ---")
    for k, v in learned_K.items():
        print(f"  {k}: {v}")

    # Build DKM tiny model
    input_dim = S * S * L
    tiny_model = build_dkm_tiny_model(student, learned_K, input_dim, n_classes, num_layers=4)
    tiny_params = sum(p.numel() for p in tiny_model.parameters())
    reduction = (1 - tiny_params / baseline_params) * 100
    print(f"\nTiny model parameters: {tiny_params:,} (reduction {reduction:.1f}% from baseline {baseline_params:,})")

    # Fine-tune tiny model
    print("\n--- Fine-tuning DKM Compressed Tiny Model ---")
    opt_ft = optim.Adam(tiny_model.parameters(), lr=0.0001, weight_decay=1e-5)
    for epoch in range(ep_fine):
        tiny_model.train()
        total_loss = 0
        for xb, yb in train_loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            opt_ft.zero_grad()
            out = tiny_model(xb)
            loss = F.cross_entropy(out, yb)
            loss.backward()
            opt_ft.step()
            total_loss += loss.item()
        if (epoch+1) % 20 == 0:
            tiny_model.eval()
            with torch.no_grad():
                out_val = tiny_model(X_test_t)
                val_acc = accuracy_score(y_test_t.cpu(), out_val.argmax(1).cpu())
            print(f"  FT Epoch {epoch+1}: loss={total_loss/len(train_loader):.4f}, val_acc={val_acc:.4f}")

    tiny_metrics = evaluate_full(tiny_model, X_test_t, y_test_t)

    # Final comparison
    print("\n" + "="*80)
    print(f"FINAL RESULTS FOR {dataset_name}")
    print("="*80)
    print(f"{'Metric':<20} {'Baseline':<18} {'Tiny DKM Compressed':<18} {'Change':<12}")
    print("-"*70)
    print(f"{'Accuracy (%)':<20} {baseline_metrics['accuracy']*100:<18.2f} {tiny_metrics['accuracy']*100:<18.2f} {(tiny_metrics['accuracy']-baseline_metrics['accuracy'])*100:>+11.2f}%")
    print(f"{'Precision (%)':<20} {baseline_metrics['precision']*100:<18.2f} {tiny_metrics['precision']*100:<18.2f}")
    print(f"{'Recall (%)':<20} {baseline_metrics['recall']*100:<18.2f} {tiny_metrics['recall']*100:<18.2f}")
    print(f"{'F1 Score (%)':<20} {baseline_metrics['f1']*100:<18.2f} {tiny_metrics['f1']*100:<18.2f}")
    print(f"{'Parameters':<20} {baseline_params:<18,} {tiny_params:<18,} {reduction:>+11.1f}%")
    print(f"{'Model Size (MB)':<20} {baseline_size_mb:<18.2f} {tiny_params*4/(1024*1024):<18.2f} {((1-tiny_params/baseline_params)*100):>+11.1f}%")
    print(f"{'Inference Time (ms)':<20} {baseline_metrics['inf_time_ms']:<18.2f} {tiny_metrics['inf_time_ms']:<18.2f} {(tiny_metrics['inf_time_ms']/baseline_metrics['inf_time_ms']-1)*100:>+11.1f}%")
    print(f"{'Training Time (s)':<20} {baseline_train_time:<18.2f} {student_train_time:<18.2f} {(student_train_time/baseline_train_time-1)*100:>+11.1f}%")



Using device: cpu


In [2]:
config =  {
        'file': 'Botswana.mat',
        'gt_file': 'Botswana_gt.mat',
        'data_key': 'Botswana',
        'gt_key': 'Botswana_gt',
        'pca_components': 30,
        'patch_size': 5,
        'description': 'Botswana Delta, 14 classes'
    }
data_path = r"Data"
dataset_name = 'Botswana'
run_experiment(data_path, dataset_name, config, 100, 120, 200)


DATASET: Botswana
PCA Components: 30, Patch Size: 5

--- Training Baseline (No Compression) ---

--- Stage 1: Training without compression loss ---
Stage1 Epoch 20/100 | Loss: 0.0010 | Val Acc: 0.9988
Early stopping at epoch 39 (best val_acc=0.9988)

--- Stage 2: Adding compression loss (K evolution) ---

--- Training Self-Adaptive Compressed Model (all layers) ---

--- Stage 1: Training without compression loss ---
Stage1 Epoch 20/120 | Loss: 0.0011 | Val Acc: 0.9938
Early stopping at epoch 35 (best val_acc=0.9951)

--- Stage 2: Adding compression loss (K evolution) ---

📊 Epoch 10/120 | λ=0.227 | Val Acc=0.9951
   Gates: {'token_embed': 0.587602436542511, 'q_0': 0.6523124575614929, 'k_0': 0.6523124575614929, 'v_0': 0.6034814119338989, 'attn_out_0': 0.6061749458312988, 'ffn1_0': 0.5605776309967041, 'ffn2_0': 0.5738946795463562, 'q_1': 0.6523124575614929, 'k_1': 0.6523124575614929, 'v_1': 0.5969991087913513, 'attn_out_1': 0.5967122316360474, 'ffn1_1': 0.5533615350723267, 'ffn2_1': 0.5

In [6]:
config =  {
        'file': 'Indian_pines_corrected.mat',
        'gt_file': 'Indian_pines_gt.mat',
        'data_key': 'indian_pines_corrected',
        'gt_key': 'indian_pines_gt',
        'pca_components': 30,
        'patch_size': 5,
        'description': 'Indian Pines, 16 classes'
    }
data_path = r"Data"
dataset_name = 'IndianPines'
run_experiment(data_path, dataset_name, config, 100, 20, 400)


DATASET: IndianPines
PCA Components: 30, Patch Size: 5

--- Training Baseline (No Compression) ---

--- Stage 1: Training without compression loss ---
Stage1 Epoch 20/100 | Loss: 0.0033 | Val Acc: 0.9664
Stage1 Epoch 40/100 | Loss: 0.0001 | Val Acc: 0.9684
Early stopping at epoch 57 (best val_acc=0.9684)

--- Stage 2: Adding compression loss (K evolution) ---

--- Training Self-Adaptive Compressed Model (all layers) ---

--- Stage 1: Training without compression loss ---
Stage1 Epoch 20/120 | Loss: 0.0161 | Val Acc: 0.9391
Stage1 Epoch 40/120 | Loss: 0.0001 | Val Acc: 0.9618
Stage1 Epoch 60/120 | Loss: 0.0001 | Val Acc: 0.9583
Early stopping at epoch 75 (best val_acc=0.9625)

--- Stage 2: Adding compression loss (K evolution) ---

📊 Epoch 10/20 | λ=1.421 | Val Acc=0.9504
   Gates: {'token_embed': 0.7413560152053833, 'q_0': 0.7977165579795837, 'k_0': 0.7977165579795837, 'v_0': 0.832544207572937, 'attn_out_0': 0.829053521156311, 'ffn1_0': 0.6213910579681396, 'ffn2_0': 0.6816086173057556

In [2]:
config =  {
        'file': 'GM01.mat',
        'gt_file': 'GM01.mat',
        'data_key': 'img',
        'gt_key': 'map',
        'pca_components': 15,
        'patch_size': 3,
        'description': 'Oil Spill Detection, 2 classes'
    }
data_path = r"Data"
dataset_name = 'GM01'
run_experiment(data_path, dataset_name, config, 50, 50, 50)


DATASET: GM01
PCA Components: 15, Patch Size: 3

--- Training Baseline (No Compression) ---

--- Stage 1: Training without compression loss ---
Stage1 Epoch 20/50 | Loss: 0.0000 | Val Acc: 1.0000
Early stopping at epoch 31 (best val_acc=1.0000)

--- Stage 2: Adding compression loss (K evolution) ---

--- Training Self-Adaptive Compressed Model (all layers) ---

--- Stage 1: Training without compression loss ---
Stage1 Epoch 20/120 | Loss: 0.0000 | Val Acc: 1.0000
Early stopping at epoch 31 (best val_acc=1.0000)

--- Stage 2: Adding compression loss (K evolution) ---

📊 Epoch 10/50 | λ=0.551 | Val Acc=1.0000
   Gates: {'token_embed': 0.9208425879478455, 'q_0': 0.9208425879478455, 'k_0': 0.9208425879478455, 'v_0': 0.9208425879478455, 'attn_out_0': 0.9208425879478455, 'ffn1_0': 0.9208425879478455, 'ffn2_0': 0.9208425879478455, 'q_1': 0.9208425879478455, 'k_1': 0.9208425879478455, 'v_1': 0.9208425879478455, 'attn_out_1': 0.9208425879478455, 'ffn1_1': 0.9208425879478455, 'ffn2_1': 0.920842

In [4]:
config =  {
        'file': 'Salinas_corrected.mat',
        'gt_file': 'Salinas_gt.mat',
        'data_key': 'salinas_corrected',
        'gt_key': 'salinas_gt',
        'pca_components': 24,
        'patch_size': 5,
        'description': 'Salinas Valley, 16 classes'
    }
data_path = r"Data"
dataset_name = 'Salinas'
run_experiment(data_path, dataset_name, config, 100, 7, 200)


DATASET: Salinas
PCA Components: 24, Patch Size: 5

--- Training Baseline (No Compression) ---

--- Stage 1: Training without compression loss ---
Stage1 Epoch 20/100 | Loss: 0.0035 | Val Acc: 0.9956
Stage1 Epoch 40/100 | Loss: 0.0000 | Val Acc: 0.9967
Stage1 Epoch 60/100 | Loss: 0.0020 | Val Acc: 0.9966
Stage1 Epoch 80/100 | Loss: 0.0029 | Val Acc: 0.9966
Stage1 Epoch 100/100 | Loss: 0.0012 | Val Acc: 0.9971

--- Stage 2: Adding compression loss (K evolution) ---

--- Training Self-Adaptive Compressed Model (all layers) ---

--- Stage 1: Training without compression loss ---
Stage1 Epoch 20/120 | Loss: 0.0002 | Val Acc: 0.9963
Stage1 Epoch 40/120 | Loss: 0.0001 | Val Acc: 0.9962
Stage1 Epoch 60/120 | Loss: 0.0001 | Val Acc: 0.9970
Early stopping at epoch 64 (best val_acc=0.9976)

--- Stage 2: Adding compression loss (K evolution) ---

📊 Epoch 7/7 | λ=3.000 | Val Acc=0.9934
   Gates: {'token_embed': 0.9110377430915833, 'q_0': 0.9355802536010742, 'k_0': 0.9355802536010742, 'v_0': 0.975

In [2]:
config =  {
        'file': 'PaviaU.mat',
        'gt_file': 'PaviaU_gt.mat',
        'data_key': 'paviaU',
        'gt_key': 'paviaU_gt',
        'pca_components': 8,
        'patch_size': 5,
        'description': 'Pavia University, 9 classes'
    }
data_path = r"Data"
dataset_name = 'PaviaU'
run_experiment(data_path, dataset_name, config, 100, 7, 200)


DATASET: PaviaU
PCA Components: 8, Patch Size: 5

--- Training Baseline (No Compression) ---

--- Stage 1: Training without compression loss ---
Stage1 Epoch 20/100 | Loss: 0.0159 | Val Acc: 0.9917
Stage1 Epoch 40/100 | Loss: 0.0086 | Val Acc: 0.9886
Stage1 Epoch 60/100 | Loss: 0.0040 | Val Acc: 0.9919
Early stopping at epoch 74 (best val_acc=0.9939)

--- Stage 2: Adding compression loss (K evolution) ---

--- Training Self-Adaptive Compressed Model (all layers) ---

--- Stage 1: Training without compression loss ---
Stage1 Epoch 20/120 | Loss: 0.0126 | Val Acc: 0.9896
Stage1 Epoch 40/120 | Loss: 0.0053 | Val Acc: 0.9887
Stage1 Epoch 60/120 | Loss: 0.0001 | Val Acc: 0.9928
Stage1 Epoch 80/120 | Loss: 0.0011 | Val Acc: 0.9851
Stage1 Epoch 100/120 | Loss: 0.0008 | Val Acc: 0.9929
Early stopping at epoch 119 (best val_acc=0.9945)

--- Stage 2: Adding compression loss (K evolution) ---

📊 Epoch 7/7 | λ=3.000 | Val Acc=0.9872
   Gates: {'token_embed': 0.7998637557029724, 'q_0': 0.912005066

In [3]:
config = {
    'file': 'Pavia.mat',
    'gt_file': 'Pavia_gt.mat',
    'data_key': 'pavia',
    'gt_key': 'pavia_gt',
    'pca_components': 15,
    'patch_size': 5,
    'description': 'Pavia Centre, 9 classes'
}
data_path = r"Data"
dataset_name = 'Pavia'
run_experiment(data_path, dataset_name, config, 100, 2, 200)


DATASET: Pavia
PCA Components: 15, Patch Size: 5

--- Training Baseline (No Compression) ---

--- Stage 1: Training without compression loss ---
Stage1 Epoch 20/100 | Loss: 0.0021 | Val Acc: 0.9989
Stage1 Epoch 40/100 | Loss: 0.0007 | Val Acc: 0.9992
Stage1 Epoch 60/100 | Loss: 0.0000 | Val Acc: 0.9991
Early stopping at epoch 64 (best val_acc=0.9995)

--- Stage 2: Adding compression loss (K evolution) ---

--- Training Self-Adaptive Compressed Model (all layers) ---

--- Stage 1: Training without compression loss ---
Stage1 Epoch 20/120 | Loss: 0.0009 | Val Acc: 0.9989
Stage1 Epoch 40/120 | Loss: 0.0003 | Val Acc: 0.9986
Stage1 Epoch 60/120 | Loss: 0.0002 | Val Acc: 0.9992
Early stopping at epoch 74 (best val_acc=0.9994)

--- Stage 2: Adding compression loss (K evolution) ---

📊 Epoch 2/2 | λ=3.000 | Val Acc=0.9984
   Gates: {'token_embed': 0.8095117807388306, 'q_0': 0.7918604612350464, 'k_0': 0.7918604612350464, 'v_0': 0.9085279703140259, 'attn_out_0': 0.9128398895263672, 'ffn1_0': 0